# EfficientNet-Based Fruit & Vegetable Classification Pipeline

This notebook implements an end-to-end image classification pipeline using the EfficientNet architecture (with configurable options from B0 to B7). The workflow covers dataset creation, model architecture design, and a two-stage training loop (frozen feature extraction followed by fine-tuning).

In [ ]:
%matplotlib inline
import sys
import json
import random
from pathlib import Path
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)

print(f'Python     : {sys.version.split()[0]}')
print(f'TensorFlow : {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs       : {[g.name for g in gpus] if gpus else "none - CPU only"}')

ROOT       = Path('.')
TRAIN_ROOT = ROOT / 'train'
TEST_ROOT  = ROOT / 'test'

EFFICIENTNET_CONFIGS = {
    'B0': {'width': 1.0, 'depth': 1.0, 'res': 224, 'dropout': 0.2},
    'B1': {'width': 1.0, 'depth': 1.1, 'res': 240, 'dropout': 0.2},
    'B2': {'width': 1.1, 'depth': 1.2, 'res': 260, 'dropout': 0.3},
    'B3': {'width': 1.2, 'depth': 1.4, 'res': 300, 'dropout': 0.3},
    'B4': {'width': 1.4, 'depth': 1.8, 'res': 380, 'dropout': 0.4},
    'B5': {'width': 1.6, 'depth': 2.2, 'res': 456, 'dropout': 0.4},
    'B6': {'width': 1.8, 'depth': 2.6, 'res': 528, 'dropout': 0.5},
    'B7': {'width': 2.0, 'depth': 3.1, 'res': 600, 'dropout': 0.5},
}

VERSION         = 'B0'
IMG_SIZE        = (EFFICIENTNET_CONFIGS[VERSION]['res'], EFFICIENTNET_CONFIGS[VERSION]['res'])
BATCH_SIZE      = 32
SEED            = 42
LR_HEAD         = 1e-3     # Phase 1: head only
LR_FINE         = 1e-4     # Phase 2: fine-tune
EPOCHS_HEAD     = 20
EPOCHS_FINE     = 30
UNFREEZE_LAYERS = 30       # unfreeze base_model layers from the end of the backbone
DROPOUT         = EFFICIENTNET_CONFIGS[VERSION]['dropout']
IMG_EXTS        = {'.jpg', '.jpeg', '.png', '.gif', '.bmp'}

tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

assert TRAIN_ROOT.exists(), 'train/ not found - run from project root'
assert TEST_ROOT.exists(),  'test/  not found - run from project root'

class EfficientNetBuilder:
    """Build and configure EfficientNet models for classification."""

    def __init__(self, version: str = 'B0', pretrained: bool = True):
        if version not in EFFICIENTNET_CONFIGS:
            raise ValueError(f"Version must be one of {list(EFFICIENTNET_CONFIGS.keys())}")

        self.version = version
        self.config = EFFICIENTNET_CONFIGS[version]
        self.pretrained = pretrained
        self.img_size = (self.config['res'], self.config['res'])

    def build(self, n_classes: int,
              include_top: bool = True,
              pooling: str = 'avg') -> models.Model:

        efficient_net_map = {
            'B0': keras.applications.EfficientNetB0,
            'B1': keras.applications.EfficientNetB1,
            'B2': keras.applications.EfficientNetB2,
            'B3': keras.applications.EfficientNetB3,
            'B4': keras.applications.EfficientNetB4,
            'B5': keras.applications.EfficientNetB5,
            'B6': keras.applications.EfficientNetB6,
            'B7': keras.applications.EfficientNetB7,
        }

        weights = 'imagenet' if self.pretrained else None
        base_model = efficient_net_map[self.version](
            input_shape=(*self.img_size, 3),
            weights=weights,
            include_top=False,
            pooling=pooling
        )

        base_model.trainable = False

        if include_top:
            model = models.Sequential([
                layers.Input(shape=(*self.img_size, 3)),
                base_model,
                layers.Dropout(self.config['dropout']),
                layers.Dense(256, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-4)),
                layers.Dropout(self.config['dropout']),
                layers.Dense(n_classes, activation='softmax')
            ])
        else:
            model = base_model

        return model

    def unfreeze_backbone(self, model: models.Model, num_layers: int = 50):
        if isinstance(model.layers[0], models.Model):
            base_model = model.layers[0]
        else:
            base_model = model

        base_model.trainable = True

        for layer in base_model.layers[:-num_layers]:
            layer.trainable = False

    def get_input_size(self) -> tuple:
        return self.img_size

## 1. Data Loading and Preprocessing Utilities

The functions below index the directory, map directory categories to class labels, and wrap paths in a high-performance `tf.data.Dataset` pipeline. Optional data augmentation functions are provided for the training dataset.

In [12]:
def discover_classes(root: Path) -> dict:
    """Flatten fruit/vegetable grouping layer -> {class_name: path}."""
    classes = {}
    for cat_dir in sorted(root.iterdir()):
        if not cat_dir.is_dir():
            continue
        for cls_dir in sorted(cat_dir.iterdir()):
            if cls_dir.is_dir():
                classes[cls_dir.name.lower()] = cls_dir
    return classes

def build_file_list(root: Path, class_to_idx: dict) -> tuple:
    """Return (file_path_strings, int_labels) from the nested fruit/veg structure."""
    paths, labels = [], []
    for cat_dir in sorted(root.iterdir()):
        if not cat_dir.is_dir():
            continue
        for cls_dir in sorted(cat_dir.iterdir()):
            if not cls_dir.is_dir():
                continue
            cls = cls_dir.name.lower()
            if cls not in class_to_idx:
                continue
            idx = class_to_idx[cls]
            for f in cls_dir.iterdir():
                if f.suffix.lower() in IMG_EXTS:
                    paths.append(str(f))
                    labels.append(idx)
    return paths, labels

def make_tf_dataset(file_paths, int_labels, n_classes,
                    img_size=(224, 224), batch_size=32,
                    augment=False, shuffle=True, seed=42):
    """Batched tf.data.Dataset. Outputs float32 images in [0, 1]."""

    def load(path, label):
        raw = tf.io.read_file(path)
        img = tf.image.decode_image(raw, channels=3, expand_animations=False)
        img.set_shape([None, None, 3])
        img = tf.image.resize(img, img_size)
        img = tf.cast(img, tf.float32)
        return img, tf.one_hot(label, n_classes)

    def augment_fn(img, label):
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_brightness(img, max_delta=0.15)
        img = tf.image.random_contrast(img, lower=0.85, upper=1.15)
        img = tf.image.random_saturation(img, lower=0.85, upper=1.15)
        img = tf.image.random_hue(img, max_delta=0.1)
        img = tf.clip_by_value(img, 0.0, 255.0)
        return img, label

    ds = tf.data.Dataset.from_tensor_slices((file_paths, int_labels))
    if shuffle:
        ds = ds.shuffle(len(file_paths), seed=seed)
    ds = ds.map(load, num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        ds = ds.map(augment_fn, num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

train_classes = discover_classes(TRAIN_ROOT)
CLASS_NAMES   = sorted(train_classes.keys())
N_CLASSES     = len(CLASS_NAMES)
CLASS_TO_IDX  = {c: i for i, c in enumerate(CLASS_NAMES)}

all_paths, all_labels = build_file_list(TRAIN_ROOT, CLASS_TO_IDX)
train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_paths, all_labels,
    test_size=0.2, stratify=all_labels, random_state=SEED
)
test_paths, test_labels = build_file_list(TEST_ROOT, CLASS_TO_IDX)

train_ds = make_tf_dataset(train_paths, train_labels, N_CLASSES,
                           img_size=IMG_SIZE, batch_size=BATCH_SIZE,
                           augment=True,  shuffle=True,  seed=SEED)
val_ds   = make_tf_dataset(val_paths,   val_labels,   N_CLASSES,
                           img_size=IMG_SIZE, batch_size=BATCH_SIZE,
                           augment=False, shuffle=False)
test_ds  = make_tf_dataset(test_paths,  test_labels,  N_CLASSES,
                           img_size=IMG_SIZE, batch_size=BATCH_SIZE,
                           augment=False, shuffle=False)

print(f'Classes     : {N_CLASSES}')
print(f'Train imgs  : {len(train_paths)}')
print(f'Val   imgs  : {len(val_paths)}')
print(f'Test  imgs  : {len(test_paths)}')

# Smoke-test one batch
for x, y in train_ds.take(1):
    print(f'Batch shape : {x.shape}  labels: {y.shape}')
    print(f'Pixel range : [{x.numpy().min():.3f}, {x.numpy().max():.3f}]')

Classes     : 36
Train imgs  : 2492
Val   imgs  : 623
Test  imgs  : 359
Batch shape : (32, 224, 224, 3)  labels: (32, 36)
Pixel range : [0.000, 255.000]


## 2. Model Architecture Builder

This class builds the model using Keras' pre-trained EfficientNet backbones. It initially freezes the backbone to train the custom dense classification layers. It also includes helper methods to unfreeze a specified number of top layers for later fine-tuning.
```
Input (224, 224, 3) [0, 1]
└- EfficientNetB0 base → (7, 7, 1280) (using pooling='avg' output)
└- Dropout(0.2)
└- Dense(256, relu + L2 regularization)
└- Dropout(0.2)
└- Dense(36, softmax)
```

In [13]:
# Construct model using EfficientNetBuilder
builder = EfficientNetBuilder(version=VERSION, pretrained=True)
model = builder.build(n_classes=N_CLASSES, include_top=True)

model.summary(show_trainable=True)

print(f'\nTrainable params     : {sum(tf.size(w).numpy() for w in model.trainable_weights):,}')
print(f'Non-trainable params : {sum(tf.size(w).numpy() for w in model.non_trainable_weights):,}')

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━┓
┃ Layer (type)                ┃ Output Shape          ┃    Param # ┃ Trai… ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━┩
│ efficientnetb0 (Functional) │ (None, 1280)          │  4,049,571 │   N   │
├-----------------------------┼-----------------------┼------------┼-------┤
│ dropout_2 (Dropout)         │ (None, 1280)          │          0 │   -   │
├-----------------------------┼-----------------------┼------------┼-------┤
│ dense_2 (Dense)             │ (None, 256)           │    327,936 │   Y   │
├-----------------------------┼-----------------------┼------------┼-------┤
│ dropout_3 (Dropout)         │ (None, 256)           │          0 │   -   │
├-----------------------------┼-----------------------┼------------┼-------┤
│ dense_3 (Dense)             │ (None, 36)            │      9,252 │   Y   │
└-----------------------------┴-----------------------┴------------┴-------┘

 Total params: 4,386,759 (16.73 MB)

 Trainable params: 337,188 (1.29 MB)

 Non-trainable params: 4,049,571 (15.45 MB)


Trainable params     : 337,188
Non-trainable params : 4,049,571


In [14]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR_HEAD),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_ph1 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=10,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=5,
        min_lr=1e-7, verbose=1
    ),
]

print('=== Phase 1: training head only ===')
history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_HEAD,
    callbacks=callbacks_ph1,
    verbose=1
)

val_acc_ph1 = max(history1.history['val_accuracy'])
print(f'\nBest val accuracy (Phase 1): {val_acc_ph1:.4f}')

=== Phase 1: training head only ===
Epoch 1/20


78/78 ━━━━━━━━━━━━━━━━━━━━ 55s 525ms/step - accuracy: 0.5594 - loss: 1.7624 - val_accuracy: 0.8058 - val_loss: 0.6935 - learning_rate: 0.0010
Epoch 2/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 36s 451ms/step - accuracy: 0.7741 - loss: 0.7487 - val_accuracy: 0.8250 - val_loss: 0.5777 - learning_rate: 0.0010
Epoch 3/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 37s 472ms/step - accuracy: 0.8367 - loss: 0.5668 - val_accuracy: 0.8299 - val_loss: 0.5709 - learning_rate: 0.0010
Epoch 4/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 38s 487ms/step - accuracy: 0.8680 - loss: 0.4647 - val_accuracy: 0.8379 - val_loss: 0.5388 - learning_rate: 0.0010
Epoch 5/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 38s 490ms/step - accuracy: 0.8856 - loss: 0.4169 - val_accuracy: 0.8507 - val_loss: 0.5168 - learning_rate: 0.0010
Epoch 6/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 40s 518ms/step - accuracy: 0.8985 - loss: 0.3659 - val_accuracy: 0.8459 - val_loss: 0.5329 - learning_rate: 0.0010
Epoch 7/20
78/78 ━━━━━━━━━━━━━━━━━━━━ 40s 509ms/step - accuracy: 0.9017 - loss: 0.3493 - val_

## 4. Phase 2 - Fine-tuning

Unfreeze the last `UNFREEZE_LAYERS` (30 layers) of the baseline EfficientNet model structure using the builder, then resume optimization at a lower learning rate.

In [ ]:
# Apply unfreeze function on backbone layers
builder.unfreeze_backbone(model, num_layers=UNFREEZE_LAYERS)

base_model = model.layers[0]
trainable_count   = sum(1 for l in base_model.layers if l.trainable)
untrainable_count = sum(1 for l in base_model.layers if not l.trainable)
print(f'Base model  - trainable: {trainable_count},  frozen: {untrainable_count}')
print(f'Fine-tuning layers [-{UNFREEZE_LAYERS}:]')
for layer in base_model.layers[-UNFREEZE_LAYERS:]:
    print(f'  {layer.name}')

# Must recompile model to apply changes to trainable weights
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR_FINE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_ph2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=10,
        restore_best_weights=True, verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=5,
        min_lr=1e-7, verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        'efficientnet_best.keras',
        monitor='val_accuracy', save_best_only=True, verbose=0
    ),
]

print('=== Phase 2: fine-tuning ===')
history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_FINE,
    initial_epoch=len(history1.history['accuracy']),
    callbacks=callbacks_ph2,
    verbose=1
)

val_acc_ph2 = max(history2.history['val_accuracy'])
print(f'\nBest val accuracy (Phase 2): {val_acc_ph2:.4f}')

## 5. Training History

Observe progression curves combining metric output logs from both baseline feature extraction and subsequent structural fine-tuning phases.

In [ ]:
acc      = history1.history['accuracy']     + history2.history['accuracy']
val_acc  = history1.history['val_accuracy'] + history2.history['val_accuracy']
loss     = history1.history['loss']         + history2.history['loss']
val_loss = history1.history['val_loss']     + history2.history['val_loss']
p1_end   = len(history1.history['accuracy'])
epochs   = range(1, len(acc) + 1)

PLOTS_DIR = Path('plots')
PLOTS_DIR.mkdir(exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, train_vals, val_vals, title, ylabel in zip(
    axes,
    [acc,  loss],
    [val_acc, val_loss],
    ['Accuracy', 'Loss'],
    ['accuracy', 'loss']
):
    ax.plot(epochs, train_vals, label='train')
    ax.plot(epochs, val_vals,   label='val')
    ax.axvline(x=p1_end + 0.5, color='grey', linestyle='--', linewidth=1,
               label='Phase 1 → 2')
    ax.set(title=title, xlabel='Epoch', ylabel=ylabel)
    ax.legend()

plt.suptitle('EfficientNetB0 - Training History', fontsize=13)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'efficientnet_history.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Test Set Evaluation

Verify generalizability boundaries using unseen test partitions. This block generates a raw classification accuracy score alongside class-by-class metrics.

In [ ]:
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f'Test loss     : {test_loss:.4f}')
print(f'Test accuracy : {test_acc:.4f}  ({test_acc*100:.2f}%)')

y_true = np.concatenate(
    [np.argmax(lbl.numpy(), axis=1) for _, lbl in test_ds]
)
y_pred = np.argmax(model.predict(test_ds, verbose=0), axis=1)

print('=== Classification Report ===')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3))

cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(20, 18))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    linewidths=0.4, ax=ax, annot_kws={'size': 7}
)
ax.set(xlabel='Predicted', ylabel='True',
       title='Confusion Matrix - EfficientNetB0')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0,  fontsize=8)
plt.tight_layout()
plt.savefig(PLOTS_DIR / 'efficientnet_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Save Model

Write completed weights and class mappings to disk to prepare for deployment or comparison.

In [ ]:
model.save('efficientnet.keras')
print('Model saved → efficientnet.keras')

# Save class names so 04_comparison.ipynb can load them
with open('class_names.json', 'w') as f:
    json.dump(CLASS_NAMES, f, indent=2)
print('Class names saved → class_names.json')

# Usage reminder
print('\nTo reload:')
print('  model = tf.keras.models.load_model("efficientnet.keras")')
print('  # model expects float32 images in [0, 255] of shape (res, res, 3)')

Model saved → efficientnet_noha.keras
Class names saved → class_names.json

To reload:
  model = tf.keras.models.load_model("efficientnet.keras")
  # model expects float32 images in [0, 1]


## Summary

| Item | Value |
|------|-------|
| Architecture | EfficientNetB0 (ImageNet weights) |
| Input | 224 × 224 × 3, float32 in [0, 255] |
| Preprocessing in model | None (Pre-trained normalization matches [0, 255] range) |
| Classes | 36 |
| Head | Sequential wrapper [GAP (Base output) → Dropout(0.2) → Dense(256, relu + L2) → Dropout(0.2) → Dense(36, softmax)] |
| Phase 1 | Frozen base, Adam(1e-3), up to 20 epochs |
| Phase 2 | Unfreeze last 30 layers, Adam(1e-4), up to 30 epochs |
| Augmentation | flip, brightness, contrast, saturation, hue |
| Saved model | `efficientnet.keras` (accepts [0,255] input) |